# 05 · Two feeds, one ledger: evidence and fix for the balance freeze

**Decision under test:** the provider's balance `quantity` is not today's holding. It is the quantity of the lot's first movement, written once and never updated.

**The fix:** replay the movement receipts, let the replay audit itself, and publish the result next to the reported number with a flag. The wealth consumer picks a number by flag.

Every query below reads the provider's raw files in `data/raw/`, never our models. So the defect sits in the data as delivered, and our pipeline is ruled out.

Scope: the deep dive covers variable incomes, where receipts carry share quantities and the defect is easiest to prove. Section 2 shows the same freeze in all five product families.

| section | question it answers |
|---|---|
| 1 | Do the two feeds actually contradict each other? |
| 2 | Is it a pattern, and what is its exact shape? |
| 3 | Does the fix work, and how does it protect itself? |
| 4, 5 | What about the two side defects (lost dates, R$ 0.00 values)? |
| 6 | What lands in the repo, and where? |


In [1]:
import os
import duckdb
import pandas as pd

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

con = duckdb.connect()

POSITIONS = "'data/raw/raw_positions.parquet'"
TRANSACTIONS = "'data/raw/raw_transactions.parquet'"

# One lot = one investment_id. These frame tables feed every section below.
con.sql(f"""
    CREATE TABLE lots AS
    SELECT DISTINCT investment_id
    FROM {POSITIONS}
    WHERE investment_type = 'VARIABLE_INCOMES'
""")

con.sql(f"""
    CREATE TABLE names AS
    SELECT
        investment_id,
        any_value(json_extract_string(payload_json, '$.data.ticker')) AS ticker,
        any_value(institution_name)                                   AS institution,
        any_value(left(party_id, 8))                                  AS party
    FROM {POSITIONS}
    WHERE investment_type = 'VARIABLE_INCOMES' AND payload_kind = 'detail'
    GROUP BY 1
""")

con.sql(f"""
    CREATE TABLE balances AS
    SELECT
        lots.investment_id,
        raw.snapshot_id,
        raw.snapshot_created_at,
        CAST(json_extract_string(raw.payload_json, '$.data.quantity')            AS DOUBLE) AS qty,
        CAST(json_extract_string(raw.payload_json, '$.data.grossAmount.amount')  AS DOUBLE) AS gross,
        CAST(json_extract_string(raw.payload_json, '$.data.closingPrice.amount') AS DOUBLE) AS close_px
    FROM {POSITIONS} AS raw
    JOIN lots USING (investment_id)
    WHERE raw.payload_kind = 'balances'
""")

con.sql("""
    CREATE TABLE latest AS
    SELECT * FROM balances
    QUALIFY row_number() OVER (
        PARTITION BY investment_id
        ORDER BY snapshot_created_at DESC, snapshot_id DESC
    ) = 1
""")

# Receipts arrive up to N times; dedup by transactionId within the lot.
con.sql(f"""
    CREATE TABLE receipts AS
    SELECT
        lots.investment_id,
        json_extract_string(trx.transaction_json, '$.transactionId') AS transaction_id,
        any_value(json_extract_string(trx.transaction_json, '$.transactionDate')) AS tx_date,
        any_value(json_extract_string(trx.transaction_json, '$.type'))            AS direction,
        any_value(json_extract_string(trx.transaction_json, '$.transactionType')) AS ttype,
        any_value(CAST(json_extract_string(trx.transaction_json, '$.transactionQuantity') AS DOUBLE)) AS qty,
        count(*) AS times_delivered
    FROM {TRANSACTIONS} AS trx
    JOIN lots USING (investment_id)
    GROUP BY 1, 2
""")

# The replay: signed net per day, then a running sum per lot.
# ENTRADA adds, SAIDA subtracts, cash-only receipts (qty NULL) count zero.
# Lost-date receipts (1970-01-01) cannot be placed on the timeline. The running
# total gives them the benefit of the doubt: lost-date buys sort first (1970
# already does), lost-date sells sort last. The final sum ignores order anyway.
con.sql("""
    CREATE TABLE replay AS
    SELECT
        investment_id,
        sum(day_net)        AS replay_qty,
        min(running_total)  AS min_running_total,
        sum(n_qty_receipts) AS n_qty_receipts
    FROM (
        SELECT *, sum(day_net) OVER (
            PARTITION BY investment_id ORDER BY eff_date
        ) AS running_total
        FROM (
            SELECT
                investment_id,
                CASE WHEN tx_date LIKE '1970%' AND direction = 'SAIDA'
                     THEN '9999-12-31' ELSE tx_date END AS eff_date,
                sum(CASE WHEN qty IS NULL THEN 0
                         WHEN direction = 'ENTRADA' THEN qty
                         ELSE -qty END)           AS day_net,
                count(*) FILTER (qty IS NOT NULL) AS n_qty_receipts
            FROM receipts
            GROUP BY 1, 2
        )
    )
    GROUP BY 1
""")

con.sql("""
    SELECT
        (SELECT count(*) FROM lots)     AS lots,
        (SELECT count(*) FROM balances) AS balance_snapshots,
        (SELECT count(*) FROM receipts) AS receipts,
        (SELECT count(*) FROM replay WHERE n_qty_receipts >= 1) AS lots_with_qty_receipts
""").df()

,lots,balance_snapshots,receipts,lots_with_qty_receipts
0,1611,17570,4747,1611


In [2]:
# Safety of the dedup: every redelivery of a transactionId is byte-identical,
# so any_value() loses nothing.
conflicting = con.sql(f"""
    SELECT count(*) FROM (
        SELECT
            lots.investment_id,
            json_extract_string(trx.transaction_json, '$.transactionId') AS transaction_id
        FROM {TRANSACTIONS} AS trx
        JOIN lots USING (investment_id)
        GROUP BY 1, 2
        HAVING count(DISTINCT trx.transaction_json::VARCHAR) > 1
    )
""").fetchone()[0]
assert conflicting == 0, conflicting
print('0 conflicting copies: dedup by transactionId is lossless')

0 conflicting copies: dedup by transactionId is lossless


## 1 · The contradiction, in one lot

Customer `9e22a589` holds ITUB4 at C6 Bank. Both feeds answer the same question: how many shares does this customer hold?


In [3]:
ITUB4 = con.sql("""
    SELECT investment_id FROM names
    WHERE party = '9e22a589' AND ticker = 'ITUB4' AND institution = 'C6 Bank'
""").fetchone()[0]

con.sql(f"""
    SELECT snapshot_created_at::DATE AS snapshot_date, qty, close_px, gross
    FROM balances
    WHERE investment_id = '{ITUB4}'
    ORDER BY snapshot_created_at, snapshot_id
""").df()

,snapshot_date,qty,close_px,gross
0,2026-07-30,651.0,24.35,15849.25
1,2026-08-07,651.0,25.72,16741.77
2,2026-08-11,651.0,24.50,15948.85
3,2026-08-11,651.0,24.50,15948.85
4,2026-08-16,651.0,24.82,16156.52
5,2026-08-21,651.0,24.20,15750.94


In [4]:
con.sql(f"""
    SELECT
        tx_date, direction, ttype, qty, times_delivered,
        sum(CASE WHEN qty IS NULL THEN 0
                 WHEN direction = 'ENTRADA' THEN qty
                 ELSE -qty END) OVER (ORDER BY tx_date, transaction_id) AS running_total
    FROM receipts
    WHERE investment_id = '{ITUB4}'
    ORDER BY tx_date, transaction_id
""").df()

,tx_date,direction,ttype,qty,times_delivered,running_total
0,2021-01-21,ENTRADA,COMPRA,651.0,6,651.0
1,2025-09-15,SAIDA,OUTROS,NaN,6,651.0
2,2025-10-30,SAIDA,VENDA,34.0,6,617.0
3,2026-05-29,ENTRADA,OUTROS,NaN,6,617.0


Every snapshot reports the same quantity. The receipts reach a different number: the opening buy minus the later sell. The closing price moves in every snapshot, so the feed is alive. Only the quantity is dead, and it equals the first receipt exactly. Both numbers cannot be today's position.


## 2 · The pattern: the quantity never moves, and equals the first receipt

Three measurements, from narrow to wide.


In [5]:
# 2a. Across snapshots, does any lot ever change quantity? Does the price change?
freeze = con.sql("""
    SELECT
        count(*)                    AS lots,
        count(*) FILTER (n_qty > 1) AS lots_where_qty_changes,
        count(*) FILTER (n_px > 1)  AS lots_where_price_changes
    FROM (
        SELECT investment_id, count(DISTINCT qty) AS n_qty, count(DISTINCT close_px) AS n_px
        FROM balances
        GROUP BY 1
    )
""").df()
assert freeze.at[0, 'lots_where_qty_changes'] == 0
freeze

,lots,lots_where_qty_changes,lots_where_price_changes
0,1611,0,1611


In [6]:
# 2b. Does the frozen number equal the lot's first receipt, digit for digit?
con.sql("""
    CREATE TABLE first_receipt AS
    SELECT investment_id, qty AS first_qty
    FROM receipts
    WHERE qty IS NOT NULL
    QUALIFY row_number() OVER (
        PARTITION BY investment_id ORDER BY tx_date, transaction_id
    ) = 1
""")

con.sql("""
    SELECT
        count(*) AS lots,
        count(*) FILTER (abs(lat.qty - fir.first_qty) < 1e-9) AS balance_equals_first_receipt,
        round(100.0 * count(*) FILTER (abs(lat.qty - fir.first_qty) < 1e-9) / count(*), 1) AS pct
    FROM latest AS lat
    JOIN first_receipt AS fir USING (investment_id)
""").df()

,lots,balance_equals_first_receipt,pct
0,1611,1578,98.0


In [7]:
# 2c. Is the freeze specific to variable incomes? Same test in all five families.
FAMILIES = {
    'BANK_FIXED_INCOMES':   ('quantity',      'transactionQuantity',      'transactionDate'),
    'CREDIT_FIXED_INCOMES': ('quantity',      'transactionQuantity',      'transactionDate'),
    'FUNDS':                ('quotaQuantity', 'transactionQuotaQuantity', 'transactionConversionDate'),
    'TREASURE_TITLES':      ('quantity',      'transactionQuantity',      'transactionDate'),
    'VARIABLE_INCOMES':     ('quantity',      'transactionQuantity',      'transactionDate'),
}

rows = []
for family, (bal_field, qty_field, date_field) in FAMILIES.items():
    rows.append(con.sql(f"""
        WITH rec AS (
            SELECT
                investment_id,
                json_extract_string(transaction_json, '$.transactionId') AS transaction_id,
                any_value(json_extract_string(transaction_json, '$.{date_field}')) AS tx_date,
                any_value(CAST(json_extract_string(transaction_json, '$.{qty_field}') AS DOUBLE)) AS qty
            FROM {TRANSACTIONS}
            WHERE investment_type = '{family}'
            GROUP BY 1, 2
        ),
        fir AS (
            SELECT
                investment_id,
                qty AS first_qty,
                count(*) OVER (PARTITION BY investment_id) AS n_qty_receipts
            FROM rec
            WHERE qty IS NOT NULL
            QUALIFY row_number() OVER (
                PARTITION BY investment_id ORDER BY tx_date, transaction_id
            ) = 1
        ),
        lat AS (
            SELECT
                investment_id,
                CAST(json_extract_string(payload_json, '$.data.{bal_field}') AS DOUBLE) AS qty
            FROM {POSITIONS}
            WHERE investment_type = '{family}' AND payload_kind = 'balances'
            QUALIFY row_number() OVER (
                PARTITION BY investment_id
                ORDER BY snapshot_created_at DESC, snapshot_id DESC
            ) = 1
        ),
        moving AS (
            SELECT investment_id
            FROM {POSITIONS}
            WHERE investment_type = '{family}' AND payload_kind = 'balances'
            GROUP BY 1
            HAVING count(DISTINCT CAST(json_extract_string(payload_json, '$.data.{bal_field}') AS DOUBLE)) > 1
        )
        SELECT
            '{family}' AS family,
            (SELECT count(*) FROM lat)    AS lots,
            (SELECT count(*) FROM moving) AS lots_where_qty_changes,
            count(*) FILTER (abs(lat.qty - fir.first_qty) < 1e-9)                          AS frozen_at_first_receipt,
            count(*) FILTER (abs(lat.qty - fir.first_qty) < 1e-9 AND fir.n_qty_receipts >= 2) AS frozen_despite_later_receipts
        FROM lat
        JOIN fir USING (investment_id)
    """).df())

families = pd.concat(rows, ignore_index=True)
assert families['lots_where_qty_changes'].sum() == 0
families

,family,lots,lots_where_qty_changes,frozen_at_first_receipt,frozen_despite_later_receipts
0,BANK_FIXED_INCOMES,6303,0,6118,3382
1,CREDIT_FIXED_INCOMES,504,0,504,254
2,FUNDS,762,0,699,431
3,TREASURE_TITLES,822,0,791,449
4,VARIABLE_INCOMES,1611,0,1578,712


Not one lot in any family ever changes quantity between snapshots. The balance is the first line of the receipts ledger, written once and never advanced. There is one ledger. A sum that stopped can be finished: that is the fix.


## 3 · The fix: replay the receipts, and let the replay audit itself

1. **Deduplicate** receipts by `transactionId`. Copies are byte-identical (asserted in section 0), so nothing is lost.
2. **Sum with sign**: ENTRADA adds, SAIDA subtracts, cash-only receipts (dividends, JCP) count zero.
3. **Audit**: a sold share must have been bought first. Replay day by day, and give lost-date receipts their most favorable placement (buys first, sells last). If the running total still goes below zero at any day's end, no ordering of the receipts can make the ledger feasible. A buy receipt is missing and the sum cannot be trusted. Verdict `movements:incomplete`: keep the reported balance and record since when it is unproven.
4. If the replay **equals the balance** (tolerance 0.001), the feeds agree. Verdict `verified`.
5. Otherwise the balance is **stale**. Verdict `stale:quantity`: publish the replayed quantity next to the reported one. Consumers use the replay.


In [8]:
con.sql("""
    CREATE TABLE verdicts AS
    SELECT
        lat.investment_id,
        lat.qty AS balance_qty,
        rep.replay_qty,
        rep.min_running_total,
        rep.n_qty_receipts,
        fir.first_qty,
        CASE
            WHEN rep.min_running_total < -0.001        THEN 'movements:incomplete'
            WHEN abs(lat.qty - rep.replay_qty) < 0.001 THEN 'verified'
            ELSE 'stale:quantity'
        END AS verdict
    FROM latest AS lat
    JOIN replay AS rep USING (investment_id)
    JOIN first_receipt AS fir USING (investment_id)
""")

summary = con.sql("""
    SELECT
        verdict,
        count(*) AS lots,
        count(*) FILTER (abs(balance_qty - first_qty) < 1e-9) AS balance_equals_first_receipt,
        count(*) FILTER (n_qty_receipts >= 2)                 AS lots_with_2plus_receipts
    FROM verdicts
    GROUP BY 1
    ORDER BY lots DESC
""").df()
assert summary['lots'].sum() == con.sql('SELECT count(*) FROM lots').fetchone()[0]
summary

,verdict,lots,balance_equals_first_receipt,lots_with_2plus_receipts
0,verified,866,866,0
1,stale:quantity,738,705,738
2,movements:incomplete,7,7,7


In [9]:
# The decisive split: does the balance survive a second trade?
survival = con.sql("""
    SELECT
        CASE WHEN n_qty_receipts = 1 THEN '1 (opening buy only)' ELSE '2 or more' END AS qty_receipts,
        count(*)                               AS lots,
        count(*) FILTER (verdict = 'verified') AS verified
    FROM verdicts
    GROUP BY 1
    ORDER BY 1
""").df()
assert survival.loc[survival['qty_receipts'] == '2 or more', 'verified'].iat[0] == 0
survival

,qty_receipts,lots,verified
0,1 (opening buy only),866,866
1,2 or more,745,0


Every verified lot has exactly one quantity receipt. Nothing happened after the opening buy, so the frozen balance is accidentally correct. Among lots with a second receipt, zero are verified. The balance has never once tracked a second trade. The `verified` bucket does not prove the balance feed works; it proves the freeze rule holds everywhere.


In [10]:
# The two unit lots, through the rule. ITUB4 from section 1, and the worst
# incomplete case: VALE3 at Banco XP, customer c7ad008b.
VALE3 = con.sql("""
    SELECT investment_id FROM names WHERE party = 'c7ad008b' AND ticker = 'VALE3'
""").fetchone()[0]

con.sql(f"""
    SELECT nam.ticker, nam.institution, ver.balance_qty, ver.replay_qty,
           ver.min_running_total, ver.verdict
    FROM verdicts AS ver
    JOIN names AS nam USING (investment_id)
    WHERE investment_id IN ('{ITUB4}', '{VALE3}')
    ORDER BY nam.ticker
""").df()

,ticker,institution,balance_qty,replay_qty,min_running_total,verdict
0,ITUB4,C6 Bank,651.0,617.0,617.0,stale:quantity
1,VALE3,Banco XP S.A.,8415.0,-831.0,-831.0,movements:incomplete


In [11]:
# VALE3 line by line: the audit catches the missing buy.
con.sql(f"""
    SELECT
        tx_date, direction, ttype, qty,
        sum(CASE WHEN qty IS NULL THEN 0
                 WHEN direction = 'ENTRADA' THEN qty
                 ELSE -qty END) OVER (ORDER BY tx_date, transaction_id) AS running_total
    FROM receipts
    WHERE investment_id = '{VALE3}'
    ORDER BY tx_date, transaction_id
""").df()

,tx_date,direction,ttype,qty,running_total
0,2025-07-23,ENTRADA,COMPRA,8415.0,8415.0
1,2025-10-20,SAIDA,VENDA,1621.0,6794.0
2,2026-01-18,SAIDA,VENDA,4206.0,2588.0
3,2026-02-14,SAIDA,VENDA,3419.0,-831.0
4,2026-03-28,SAIDA,ALUGUEIS,NaN,-831.0
5,2026-04-19,SAIDA,JCP,NaN,-831.0


The fourth line sells more shares than the ledger holds, so the running total goes negative. No real portfolio can. A buy receipt is missing, so the final sum is also missing that buy. Publishing it would understate the position, so the rule refuses and keeps the reported balance instead. A buy older than the feed's history does not explain these lots: each starts with a recorded buy, and the same account's receipts reach back years earlier.


In [12]:
# Why the audit uses a running minimum with benefit-of-the-doubt ordering.
con.sql("""
    WITH face_value AS (
        SELECT investment_id, min(running_total) AS min_running_total
        FROM (
            SELECT *, sum(day_net) OVER (
                PARTITION BY investment_id ORDER BY tx_date
            ) AS running_total
            FROM (
                SELECT
                    investment_id,
                    tx_date,
                    sum(CASE WHEN qty IS NULL THEN 0
                             WHEN direction = 'ENTRADA' THEN qty
                             ELSE -qty END) AS day_net
                FROM receipts
                GROUP BY 1, 2
            )
        )
        GROUP BY 1
    )
    SELECT
        count(*) FILTER (ver.replay_qty < -0.001)        AS lots_ending_negative,
        count(*) FILTER (fav.min_running_total < -0.001) AS dips_at_face_value,
        count(*) FILTER (ver.min_running_total < -0.001) AS dips_under_best_case
    FROM verdicts AS ver
    JOIN face_value AS fav USING (investment_id)
""").df()

,lots_ending_negative,dips_at_face_value,dips_under_best_case
0,5,20,7


Taken at face value, 20 lots dip below zero. But 13 of them dip only because a sell with a lost date (1970-01-01) sorts to the front of the timeline. The final sum does not depend on order, so those 13 are repairable: the audit places lost-date sells last before judging. That leaves the lots that dip under every possible ordering; 5 of them even end negative. Only those are refused. A dip proves a missing buy just as hard as a negative ending, because the missing buy corrupts the final sum either way.


## 4 · Side defect: receipts dated 1970-01-01

1970-01-01 is the zero of computer clocks: a placeholder that means the real date was lost.


In [13]:
# Epoch receipts per family (funds date their receipts with transactionConversionDate).
con.sql(f"""
    SELECT investment_type AS family, count(*) AS epoch_receipts, count(DISTINCT investment_id) AS lots
    FROM (
        SELECT
            any_value(investment_type) AS investment_type,
            investment_id,
            json_extract_string(transaction_json, '$.transactionId') AS transaction_id,
            any_value(coalesce(
                json_extract_string(transaction_json, '$.transactionDate'),
                json_extract_string(transaction_json, '$.transactionConversionDate'))) AS tx_date
        FROM {TRANSACTIONS}
        GROUP BY investment_id, transaction_id
    )
    WHERE tx_date LIKE '1970%' OR tx_date LIKE '0001%'
    GROUP BY 1
    ORDER BY 2 DESC
""").df()

,family,epoch_receipts,lots
0,BANK_FIXED_INCOMES,144,144
1,VARIABLE_INCOMES,98,98
2,FUNDS,41,41


In [14]:
# In variable incomes the epoch receipt is usually the opening buy:
# its quantity equals the balance digit for digit.
con.sql("""
    SELECT
        count(*) AS epoch_receipts,
        count(DISTINCT rec.investment_id) AS lots,
        count(*) FILTER (abs(rec.qty - lat.qty) < 1e-9) AS quantity_equals_balance
    FROM receipts AS rec
    JOIN latest AS lat USING (investment_id)
    WHERE rec.tx_date LIKE '1970%'
""").df()

,epoch_receipts,lots,quantity_equals_balance
0,98,98,53


In [15]:
# Unit case: a HASH11 lot whose opening buy is epoch-dated. The receipt is real,
# only its date is the placeholder: its quantity equals the balance digit for digit.
HASH11 = con.sql("""
    SELECT rec.investment_id
    FROM receipts AS rec
    JOIN names AS nam USING (investment_id)
    JOIN latest AS lat USING (investment_id)
    WHERE nam.ticker = 'HASH11' AND rec.tx_date LIKE '1970%'
      AND rec.direction = 'ENTRADA' AND abs(rec.qty - lat.qty) < 1e-9
    LIMIT 1
""").fetchone()[0]

con.sql(f"""
    SELECT
        tx_date, direction, ttype, qty,
        sum(CASE WHEN qty IS NULL THEN 0
                 WHEN direction = 'ENTRADA' THEN qty
                 ELSE -qty END) OVER (ORDER BY tx_date, transaction_id) AS running_total
    FROM receipts
    WHERE investment_id = '{HASH11}'
    ORDER BY tx_date, transaction_id
""").df()

,tx_date,direction,ttype,qty,running_total
0,1970-01-01,ENTRADA,COMPRA,1868.0,1868.0
1,2026-07-29,ENTRADA,COMPRA,356.0,2224.0
2,2026-08-10,SAIDA,OUTROS,NaN,2224.0
3,2026-08-21,SAIDA,ALUGUEIS,NaN,2224.0


In [16]:
# Can the lost date be recovered? Bank fixed income details carry purchaseDate.
# Variable income and funds details carry no date field, so there the date stays unknown.
con.sql(f"""
    WITH epoch_lots AS (
        SELECT DISTINCT investment_id
        FROM (
            SELECT
                investment_id,
                json_extract_string(transaction_json, '$.transactionId') AS transaction_id,
                any_value(json_extract_string(transaction_json, '$.transactionDate')) AS tx_date
            FROM {TRANSACTIONS}
            WHERE investment_type = 'BANK_FIXED_INCOMES'
            GROUP BY 1, 2
        )
        WHERE tx_date LIKE '1970%' OR tx_date LIKE '0001%'
    ),
    detail AS (
        SELECT
            investment_id,
            any_value(json_extract_string(payload_json, '$.data.purchaseDate')) AS purchase_date
        FROM {POSITIONS}
        WHERE investment_type = 'BANK_FIXED_INCOMES' AND payload_kind = 'detail'
        GROUP BY 1
    )
    SELECT
        count(*) AS bank_fixed_epoch_lots,
        count(*) FILTER (
            detail.purchase_date IS NOT NULL
            AND detail.purchase_date NOT LIKE '1970%'
            AND detail.purchase_date NOT LIKE '0001%') AS recoverable_from_purchase_date
    FROM epoch_lots
    LEFT JOIN detail USING (investment_id)
""").df()

,bank_fixed_epoch_lots,recoverable_from_purchase_date
0,144,128


The quantity replay adds receipts in any order, so a lost date never blocks the fix. The audit already gives lost-date receipts their most favorable placement (section 3), so a lost date alone never causes a refusal. The pipeline already nulls placeholder dates and flags the movement `missing:transaction_date` (`clean_missing_date` in `macros/openfinance.sql`). The new action: for bank fixed incomes, restore the date from the detail's `purchaseDate`.


## 5 · Side defect: snapshots that report the position worth R$ 0.00

A snapshot reports total value zero while the same payload carries a positive quantity and a positive price. The payload contradicts its own arithmetic.


In [17]:
flap = con.sql("""
    WITH flap AS (
        SELECT * FROM balances WHERE gross = 0 AND qty > 0
    )
    SELECT
        count(*)                                 AS flap_rows,
        count(DISTINCT investment_id)            AS lots,
        count(*) FILTER (close_px > 0)           AS with_positive_price,
        min(snapshot_created_at::DATE)           AS first_seen,
        max(snapshot_created_at::DATE)           AS last_seen,
        count(*) FILTER (EXISTS (
            SELECT 1 FROM balances AS nxt
            WHERE nxt.investment_id = flap.investment_id
              AND nxt.snapshot_created_at > flap.snapshot_created_at
              AND nxt.gross > 0))                AS recovers_in_a_later_snapshot
    FROM flap
""").df()
assert flap.at[0, 'flap_rows'] == flap.at[0, 'recovers_in_a_later_snapshot']
flap

,flap_rows,lots,with_positive_price,first_seen,last_seen,recovers_in_a_later_snapshot
0,144,91,144,2026-08-06,2026-08-16,144


In [18]:
# Unit case: BPAC11 at Nubank, customer 9e22a589. One arithmetic check catches the row.
BPAC11 = con.sql("""
    SELECT DISTINCT investment_id
    FROM balances
    JOIN names USING (investment_id)
    WHERE ticker = 'BPAC11' AND institution = 'Nubank' AND party = '9e22a589' AND gross = 0
""").fetchone()[0]

con.sql(f"""
    SELECT
        snapshot_created_at::DATE AS snapshot_date,
        qty, close_px, gross,
        round(qty * close_px, 2) AS qty_times_price,
        CASE WHEN abs(gross - qty * close_px) <= 0.0002 * qty * close_px
             THEN 'pass' ELSE 'FAIL' END AS arithmetic_check
    FROM balances
    WHERE investment_id = '{BPAC11}'
    ORDER BY snapshot_created_at, snapshot_id
""").df()

,snapshot_date,qty,close_px,gross,qty_times_price,arithmetic_check
0,2026-07-29,4375.0,49.17,215127.50,215118.75,pass
1,2026-07-31,4375.0,47.82,209192.38,209212.50,pass
2,2026-08-05,4375.0,49.33,215825.75,215818.75,pass
3,2026-08-07,4375.0,48.86,213772.56,213762.50,pass
4,2026-08-08,4375.0,49.19,215185.69,215206.25,pass
5,2026-08-11,4375.0,47.98,209907.25,209912.50,pass
6,2026-08-12,4375.0,49.02,0.00,214462.50,FAIL
7,2026-08-12,4375.0,49.02,0.00,214462.50,FAIL
8,2026-08-14,4375.0,46.46,203282.19,203262.50,pass
9,2026-08-16,4375.0,47.55,208036.94,208031.25,pass


Every snapshot but the flap agrees with its own arithmetic within rounding (the price is rounded to cents). The action: reject the row at intake with the check `value = quantity x price`. The warehouse already flags these rows `zero_flap` (`holding_data_quality_flags` in `macros/openfinance.sql`). The consumer fills the gap by valuing the lot at quantity times the last non-zero price.


## 6 · Conclusion: what lands in the repo

The reported balance is never overwritten. Every derived number is published next to it with a flag, and the consumer picks by flag:

| verdict | holdings page shows |
|---|---|
| `verified` | the reported quantity |
| `stale:quantity` | the replayed quantity, flagged |
| `movements:incomplete` | the reported quantity, plus a caveat: unproven since the first negative day |

Implementation, in order:

| # | action | where |
|---|---|---|
| 1 | Lot-grain replay model: `replay_qty`, `min_running_total`, verdict. New grain (lot from movements), so a new model is justified | `models/canonical/intermediate/`, built from `int_variable_incomes_transactions` |
| 2 | Additive columns on holdings: `quantity_derived` plus the verdict in `data_quality_flags`. Contract change lands in the same commit | `models/consumption/fct_holdings.sql` + `contracts/_consumption.yml` |
| 3 | Pick per verdict (table above) | `consumers/wealth/holdings.sql` |
| 4 | Restore epoch dates from the detail's `purchaseDate` | `int_bank_fixed_incomes_transactions` |
| 5 | Value flapped rows at quantity times last non-zero price (`zero_flap` already fires) | `consumers/wealth/holdings.sql` |
| 6 | Extend the replay to the other four families: the freeze is universal (section 2c) and their receipts carry quantities too | follow-up, same pattern |
